# 05 · Attention From Scratch

Companion to **Chapter 5**. You will:
1. Build scaled dot-product attention and verify it against PyTorch's fused kernel
2. **Deliberately introduce four bugs** and see what each one does
3. Verify the √d_k claim empirically

Part 2 is the important one. None of those four bugs raises an exception, and three
of them still train to a plausible-looking loss curve.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)

## 1 · The implementation

Four steps: score → scale → mask+softmax → weighted sum.

In [ ]:
def attention(q, k, v, causal=True, scale=True, return_weights=False):
    """
    q, k: (..., T_q, d_k)
    v:    (..., T_k, d_v)
    returns (..., T_q, d_v)
    """
    d_k = q.size(-1)

    # 1. score every query against every key
    s = q @ k.transpose(-2, -1)                       # (..., T_q, T_k)

    # 2. scale
    if scale:
        s = s / math.sqrt(d_k)

    # 3. mask BEFORE softmax, with a large negative number
    if causal:
        T_q, T_k = s.shape[-2], s.shape[-1]
        block = torch.ones(T_q, T_k, dtype=torch.bool, device=s.device).triu(T_k - T_q + 1)
        s = s.masked_fill(block, torch.finfo(s.dtype).min)

    w = s.softmax(dim=-1)                             # (..., T_q, T_k) rows sum to 1

    # 4. weighted sum of values
    out = w @ v                                       # (..., T_q, d_v)
    return (out, w) if return_weights else out


B, H, T, D = 2, 4, 16, 64
q, k, v = (torch.randn(B, H, T, D) for _ in range(3))

mine = attention(q, k, v, causal=True)
ref  = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print("shapes match:", mine.shape == ref.shape, tuple(mine.shape))
print("max abs diff:", (mine - ref).abs().max().item())
assert torch.allclose(mine, ref, atol=1e-5), "does not match PyTorch"
print("MATCHES PyTorch's fused kernel ✓")

In [ ]:
# Look at the weights themselves
_, w = attention(q, k, v, causal=True, return_weights=True)
print("weights shape:", tuple(w.shape), " <- (B, H, T, T), quadratic in T")
print("row sums (should all be 1):", w.sum(-1)[0, 0, :5].tolist())
print("row 0 (can only see itself):", w[0, 0, 0, :4].tolist())
print("row 3 (sees 0..3):          ", w[0, 0, 3, :6].tolist())

assert torch.allclose(w.sum(-1), torch.ones(B, H, T), atol=1e-6)
assert (w[..., :, :].triu(1).abs() < 1e-9).all(), "future must be EXACTLY zero"
print("\nmasked positions are exactly 0.0 ✓  (that is what -inf-before-softmax buys)")

## 2 · The four bugs

Each of these runs without error. Watch what changes.

In [ ]:
def entropy(w):
    """Average entropy of the attention rows, in bits. Low = peaked/saturated."""
    p = w.clamp_min(1e-12)
    return float(-(p * p.log2()).sum(-1).mean())

# --- BUG 1: forget the /sqrt(d_k) --------------------------------------
_, w_ok  = attention(q, k, v, scale=True,  return_weights=True)
_, w_bad = attention(q, k, v, scale=False, return_weights=True)
print("BUG 1 -- no scaling")
print(f"  entropy  scaled {entropy(w_ok):.3f} bits   unscaled {entropy(w_bad):.3f} bits")
print(f"  max weight  scaled {w_ok.max():.3f}      unscaled {w_bad.max():.3f}")
print("  -> unscaled collapses toward one-hot; softmax saturates; gradients vanish\n")

In [ ]:
# --- BUG 2: softmax over the wrong dim ---------------------------------
s = (q @ k.transpose(-2, -1)) / math.sqrt(D)
w_rows = s.softmax(dim=-1)     # correct: normalise over KEYS
w_cols = s.softmax(dim=-2)     # bug:     normalise over QUERIES
print("BUG 2 -- softmax(dim=-2)")
print(f"  correct: row sums {w_rows.sum(-1)[0,0,:3].tolist()}")
print(f"  bug:     row sums {w_cols.sum(-1)[0,0,:3].tolist()}")
print("  -> output is no longer a convex combination; magnitudes vary by position\n")

# --- BUG 3: transposed score matrix ------------------------------------
s_ok  = q @ k.transpose(-2, -1)
s_bad = k @ q.transpose(-2, -1)
print("BUG 3 -- k @ q.T instead of q @ k.T")
print(f"  is it just the transpose? {torch.allclose(s_bad, s_ok.transpose(-2,-1))}")
print("  -> row i now means 'how much everyone attends TO i'.")
print("     With a causal mask this inverts the direction of information flow.\n")

In [ ]:
# --- BUG 4: mask AFTER softmax -----------------------------------------
s = (q @ k.transpose(-2, -1)) / math.sqrt(D)
tri = torch.ones(T, T, dtype=torch.bool).triu(1)

w_before = s.masked_fill(tri, torch.finfo(s.dtype).min).softmax(-1)   # correct
w_after  = s.softmax(-1).masked_fill(tri, 0.0)                        # bug

print("BUG 4 -- mask after softmax")
print(f"  correct row sums: {w_before.sum(-1)[0,0,:6].round(decimals=3).tolist()}")
print(f"  buggy   row sums: {w_after.sum(-1)[0,0,:6].round(decimals=3).tolist()}")
print("  -> early tokens' outputs get shrunk by a POSITION-DEPENDENT factor.")
print("     Token 0 should attend 100% to itself; instead it keeps only a fraction.")

assert w_after.sum(-1)[0, 0, 0] < 0.9, "row 0 should be badly under-normalised"
print("\nNone of these four raised an exception. That is the lesson.")

## 3 · Exercise 5.2 — verify the √d_k claim

The course claims: for unit-variance q and k, `std(q·k) = sqrt(d_k)`.
That is *why* the scaling factor is what it is. Let's check it.

In [ ]:
print(f"{'d_k':>6} {'empirical std':>15} {'sqrt(d_k)':>12} {'error':>8}   "
      f"{'H unscaled':>11} {'H scaled':>10}")
for d in [4, 16, 64, 256, 1024]:
    a, b = torch.randn(40000, d), torch.randn(40000, d)
    dots = (a * b).sum(-1)
    emp, theory = dots.std().item(), math.sqrt(d)

    # what that does to a softmax over 64 keys
    scores = torch.randn(200, 64, d) @ torch.randn(200, d, 64) 
    h_un = entropy(scores.softmax(-1))
    h_sc = entropy((scores / math.sqrt(d)).softmax(-1))

    print(f"{d:>6} {emp:>15.3f} {theory:>12.3f} {abs(emp-theory)/theory:>7.1%}   "
          f"{h_un:>11.3f} {h_sc:>10.3f}")
    assert abs(emp - theory) / theory < 0.05

print("\nstd(q·k) == sqrt(d_k) ✓")
print("Unscaled entropy collapses as d_k grows. Scaled entropy stays ~constant --")
print("which is why the SAME architecture works at d_head=64 and d_head=256.")

## 4 · Cross-attention: output length follows the QUERY

The property that makes encoder–decoder models work.

In [ ]:
q_dec = torch.randn(1, 7, 64)     # decoder: 7 tokens
k_enc = torch.randn(1, 23, 64)    # encoder: 23 tokens
v_enc = torch.randn(1, 23, 64)

out = attention(q_dec, k_enc, v_enc, causal=False)
print(f"q {tuple(q_dec.shape)}  k,v {tuple(k_enc.shape)}  ->  out {tuple(out.shape)}")
assert out.shape == (1, 7, 64)
print("\nOne output vector PER QUERY, regardless of how many keys there were.")
print("That is how a decoder of any length reads an encoder of any length.")

## 5 · The quadratic, made concrete (Exercise 5.4)

In [ ]:
def fmt(b):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if b < 1024: return f"{b:7.2f} {u}"
        b /= 1024
    return f"{b:.1f} PB"

n_head, n_layer = 32, 32
print(f"Attention score matrices, bf16, n_head={n_head}:")
print(f"{'T':>8} {'per layer':>12} {'all layers':>12}")
for T_ in [1024, 4096, 16384, 32768, 131072]:
    per = n_head * T_ * T_ * 2
    print(f"{T_:>8} {fmt(per):>12} {fmt(per*n_layer):>12}")

print("\nThis is why FlashAttention is not an optimisation -- it is a prerequisite.")
print("(Chapter 17: it never materialises this matrix at all.)")

# where attention FLOPs overtake parameter FLOPs for a 7B model
N, C, L = 7e9, 4096, 32
T_cross = 2 * N / (12 * L * C)
print(f"\nFor a 7B model, attention FLOPs overtake parameter FLOPs at T ≈ {T_cross:,.0f}")

---
## Self-check

1. Why √d_k and not d_k?
2. Why must the mask be applied before the softmax, with −∞ rather than 0?
3. `q` is `(B, T_q, d)`, `k`/`v` are `(B, T_k, d)`. Output shape?
4. Which of the four bugs above would you catch from the loss curve alone?

<details><summary>Answers</summary>

1. Variances add over the d_k summed terms, so Var(q·k) = d_k and **std** = √d_k.
   Dividing by d_k would over-correct and flatten the distribution toward uniform.
2. `exp(-inf) = 0` gives exactly-zero weights **and** leaves the surviving row summing
   to 1. Zeroing after softmax leaves rows summing to <1 — a position-dependent
   shrinkage of the output.
3. `(B, T_q, d)` — one output per query.
4. Essentially none of them reliably. Bug 3 (transposed scores) leaks future
   information and would show an implausibly low loss; the rest just train worse.
   This is why you verify against a reference implementation.

</details>

**Next:** `06_multihead.ipynb`